# Assault DDQN — Entregable final HU011

Notebook final de Assault para el Reto 1. Orquesta bootstrap reproducible, entrenamiento/resume cuando sea necesario, modelo compacto, evidencias TensorBoard, evaluación final de explotación, baseline, video, reporte técnico y gate de cumplimiento del enunciado.


## 1. Bootstrap Local -> GitHub -> Colab


In [ ]:
import os

os.environ.setdefault("ASSAULT_BOOTSTRAP_REF", "feature/hu011-entregable-final-assault")
os.environ.pop("ASSAULT_BOOTSTRAP_COMMIT", None)
os.environ.setdefault("ASSAULT_TRAINING_PROFILE", "full")
os.environ.setdefault("ASSAULT_PROJECT_RUN_ID", "assault_ddqn_full_001")
os.environ.setdefault("ASSAULT_REQUESTED_MODE", "auto")
os.environ.setdefault("ASSAULT_EXECUTION_MODE", "auto")
os.environ.setdefault("ASSAULT_EVALUATION_EPISODES", "10")
os.environ.setdefault("ASSAULT_EVALUATION_EPSILON", "0.0")
os.environ.setdefault("ASSAULT_VIDEO_MAX_STEPS", "1800")
os.environ.setdefault("GLOBAL_RETO_MULTI_ALGORITHM", "PENDING")

from pathlib import Path
try:
    from google.colab import drive  # type: ignore
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/reinforcement_learning_reto_1")
else:
    BASE = Path.cwd()

os.environ.setdefault("ASSAULT_MLFLOW_TRACKING_URI", (BASE / "mlruns").as_uri())
os.environ.setdefault("ASSAULT_CHECKPOINT_DIR", str(BASE / "checkpoints"))
os.environ.setdefault("ASSAULT_TENSORBOARD_DIR", str(BASE / "tensorboard"))
print("BASE:", BASE)


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"

def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True

def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()

if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import install_project_requirements, prepare_execution_environment, verify_environment_import

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)
PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir
if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)
environment_source = verify_environment_import(bootstrap)
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)


## 2. Imports, configuración y fuentes de evidencia


In [ ]:
import json
import math
import torch
from IPython.display import Video, display

from src.agent import DDQNAgent
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.evaluator import evaluate_agent
from src.hu009c_delivery import resolve_hu009c_execution_mode
from src.hu011_delivery import (
    MANDATORY_ASSAULT_CRITERIA,
    build_delivery_gate,
    build_evaluation_artifact,
    compare_with_random_baseline,
    criteria_from_statuses,
    load_random_baseline,
    plot_exploitation_rewards,
    validate_exploitation_figure_data,
    write_json_atomic,
)
from src.model_artifact import export_inference_model, load_inference_model
from src.preflight import run_preflight_checks
from src.reporting import plot_training_figures, prepare_training_figures
from src.session_bootstrap import compute_config_fingerprint, prepare_training_session, update_experiment_state_after_success
from src.training_profiles import resolve_training_profile
from src.training_session import run_training_session
from src.utils import get_runtime_info, load_yaml_config
from src.video import generate_assault_demo_video

BASE_CONFIG = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
profile_context = resolve_training_profile(BASE_CONFIG, os.environ.get("ASSAULT_TRAINING_PROFILE", "full"))
config = profile_context.config
seed = int(config["reproducibility"]["seed"])
runtime_info = get_runtime_info()
random_baseline = load_random_baseline(ASSAULT_DIR / "data" / "baseline_random_assault.json")
full_training_evidence = json.loads((ASSAULT_DIR / "data" / "full_training_summary_assault.json").read_text(encoding="utf-8"))
print("TRAINING_PROFILE=", profile_context.name)
print("RUNTIME=", runtime_info)
print("RANDOM_BASELINE=", random_baseline.as_dict())


## 3. Entorno, preprocessing y preflight


In [ ]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_probe = create_assault_env(config, mode="eval", seed=seed + 1)
obs, info = train_env.reset(seed=seed)
env_metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)
train_env.close()
eval_probe.close()
preflight_report = run_preflight_checks(config)
if not preflight_report.ready_for_training:
    raise RuntimeError(preflight_report.format_summary())
print("READY_FOR_TRAINING=True")
print(env_metadata)


## 4. Ejecución automática: NEW / RESUME / DELIVERY


In [ ]:
PROJECT_RUN_ID = os.environ.get("ASSAULT_PROJECT_RUN_ID", "assault_ddqn_full_001")
TARGET_TIMESTEPS = int(profile_context.target_timesteps)
CHECKPOINT_STEP = TARGET_TIMESTEPS
SOURCE_CHECKPOINT_PATH = Path(os.environ.get("ASSAULT_SOURCE_CHECKPOINT_PATH") or (Path(os.environ["ASSAULT_CHECKPOINT_DIR"]) / PROJECT_RUN_ID / f"checkpoint_step_{CHECKPOINT_STEP:06d}.pt"))
EXECUTION_MODE = os.environ.get("ASSAULT_EXECUTION_MODE", "auto").strip().lower()
REQUESTED_MODE = os.environ.get("ASSAULT_REQUESTED_MODE", "auto").strip().lower()

session_kwargs = {
    "base_path": BASE,
    "project_run_id": PROJECT_RUN_ID,
    "target_timesteps": TARGET_TIMESTEPS,
    "requested_mode": REQUESTED_MODE,
    "config": config,
    "checkpoint_root": os.environ.get("ASSAULT_CHECKPOINT_DIR"),
    "tensorboard_root": os.environ.get("ASSAULT_TENSORBOARD_DIR"),
    "tracking_uri": os.environ.get("ASSAULT_MLFLOW_TRACKING_URI"),
    "resume_mode": "resume_full",
    "bootstrap_ref": BOOTSTRAP_REF,
    "bootstrap_commit": bootstrap.resolved_sha,
    "mlflow_enabled": False,
}

delivery_execution = resolve_hu009c_execution_mode(
    run_training=None,
    execution_mode=EXECUTION_MODE,
    project_run_id=PROJECT_RUN_ID,
    target_timesteps=TARGET_TIMESTEPS,
    final_checkpoint_path=SOURCE_CHECKPOINT_PATH,
    prepare_training_session_fn=prepare_training_session,
    prepare_training_session_kwargs=session_kwargs,
)
session_context = delivery_execution.session_context
RUN_TRAINING = delivery_execution.training_required
AUTO_RESOLUTION = delivery_execution.auto_resolution
TRAINING_SKIPPED_FINAL_CHECKPOINT_EXISTS = bool(AUTO_RESOLUTION == "DELIVERY" and SOURCE_CHECKPOINT_PATH.exists())
print("ASSAULT_EXECUTION_MODE=", EXECUTION_MODE)
print("AUTO_RESOLUTION=", AUTO_RESOLUTION)
print("RUN_TRAINING=", RUN_TRAINING)
print("TRAINING_SKIPPED_FINAL_CHECKPOINT_EXISTS=", TRAINING_SKIPPED_FINAL_CHECKPOINT_EXISTS)


In [ ]:
session_summary = None
if not RUN_TRAINING:
    print("TRAINING_SKIPPED=True; AUTO_RESOLUTION=", AUTO_RESOLUTION)
else:
    if not _running_in_colab() or not torch.cuda.is_available():
        raise RuntimeError("El entrenamiento full de Assault requiere Google Colab con GPU habilitada.")
    session_summary = run_training_session(
        config=config,
        checkpoint_root=session_context.checkpoint_root,
        tensorboard_root=session_context.tensorboard_root,
        run_id=session_context.project_run_id,
        repo_path=PROJECT_ROOT,
        tracking_mode=session_context.tracking_mode,
        checkpoint_input=session_context.checkpoint_input,
        resume_mode=session_context.resume_mode,
        total_timesteps=session_context.target_timesteps,
        device="cuda",
    )
    update_experiment_state_after_success(
        session_context,
        session_context.mlflow_run_id,
        session_summary.checkpoint_output_reference,
        session_summary.final_global_step,
    )
    SOURCE_CHECKPOINT_PATH = Path(session_summary.checkpoint_output_reference)
    assert session_summary.final_global_step == TARGET_TIMESTEPS
    print("TRAINING_COMPLETE=True")
    print(session_summary.as_dict())

if not SOURCE_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Final checkpoint not found: {SOURCE_CHECKPOINT_PATH}")
print("SOURCE_CHECKPOINT_READY=True")


## 12. Modelo compacto de inferencia


In [ ]:
COMPACT_MODEL_PATH = Path(os.environ.get("ASSAULT_COMPACT_MODEL_PATH") or (BASE / "models" / PROJECT_RUN_ID / "assault_ddqn_model.pt"))
compact_model_info = export_inference_model(
    checkpoint_path=SOURCE_CHECKPOINT_PATH,
    output_path=COMPACT_MODEL_PATH,
    project_run_id=PROJECT_RUN_ID,
    config=config,
    source_checkpoint_step=CHECKPOINT_STEP,
    repo_path=PROJECT_ROOT,
    extra_metadata={
        "config_fingerprint": compute_config_fingerprint(config),
        "training_summary": full_training_evidence["training"],
        "training_runtime": full_training_evidence["runtime"],
        "training_hardware": full_training_evidence["hardware"],
    },
    overwrite=True,
)
compact_agent, compact_model_info = load_inference_model(
    COMPACT_MODEL_PATH,
    device="cuda" if torch.cuda.is_available() else "cpu",
    expected_sha256=compact_model_info.sha256,
    expected_project_run_id=PROJECT_RUN_ID,
)
assert compact_model_info.size_bytes < 100 * 1024 * 1024
print("COMPACT_MODEL_READY=True")
print(compact_model_info.as_dict())


## 13. Figuras TensorBoard reales


In [ ]:
TENSORBOARD_RUN_DIR = Path(os.environ.get("ASSAULT_TENSORBOARD_RUN_DIR") or (Path(os.environ["ASSAULT_TENSORBOARD_DIR"]) / PROJECT_RUN_ID))
training_figures = prepare_training_figures(TENSORBOARD_RUN_DIR, reward_window=10)
rendered_figures = plot_training_figures(training_figures)
for figure in rendered_figures:
    display(figure)
print("training_figure_count:", len(training_figures))
print("TENSORBOARD_FIGURES_READY=True")


## 14. Evaluacion y video desde modelo compacto


In [ ]:
EVALUATION_EPISODES = max(10, int(os.environ.get("ASSAULT_EVALUATION_EPISODES", "10")))
EVALUATION_EPSILON = float(os.environ.get("ASSAULT_EVALUATION_EPSILON", "0.0"))
if EVALUATION_EPSILON != 0.0:
    raise ValueError("HU011 final evaluation requires epsilon=0.0")
FINAL_EVAL_BASE_SEED = int(os.environ.get("ASSAULT_FINAL_EVAL_BASE_SEED", "20042"))
FINAL_EVAL_SEEDS = [FINAL_EVAL_BASE_SEED + index for index in range(EVALUATION_EPISODES)]

eval_env = create_assault_env(config, mode="eval", seed=FINAL_EVAL_SEEDS[0])
try:
    compact_evaluation_summary = evaluate_agent(
        eval_env,
        compact_agent,
        episodes=EVALUATION_EPISODES,
        epsilon=EVALUATION_EPSILON,
        episode_seeds=FINAL_EVAL_SEEDS,
    )
finally:
    eval_env.close()
print(compact_evaluation_summary.as_dict())
print("EVALUATION_READY=True")

FINAL_EVALUATION_PATH = BASE / "evaluations" / PROJECT_RUN_ID / "final_compact_evaluation.json"
final_evaluation_artifact = build_evaluation_artifact(
    compact_evaluation_summary,
    compact_model_info,
    PROJECT_RUN_ID,
    SOURCE_CHECKPOINT_PATH,
    base_seed=FINAL_EVAL_BASE_SEED,
    git_sha=bootstrap.resolved_sha,
)
write_json_atomic(FINAL_EVALUATION_PATH, final_evaluation_artifact)

exploitation_figure = plot_exploitation_rewards(compact_evaluation_summary)
display(exploitation_figure)
EXPLOITATION_REWARD_FIGURE_PASS = validate_exploitation_figure_data(compact_evaluation_summary, compact_evaluation_summary.rewards)
print("EXPLOITATION_REWARD_FIGURE_PASS=", EXPLOITATION_REWARD_FIGURE_PASS)

baseline_comparison = compare_with_random_baseline(compact_evaluation_summary, random_baseline)
print("BASELINE_COMPARISON=", baseline_comparison.as_dict())

VIDEO_PATH = Path(os.environ.get("ASSAULT_VIDEO_PATH") or (BASE / "videos" / PROJECT_RUN_ID / "assault_ddqn_demo.mp4"))
video_metadata = {
    "project_run_id": PROJECT_RUN_ID,
    "source_checkpoint_step": CHECKPOINT_STEP,
    "model_sha256": compact_model_info.sha256,
    "epsilon": EVALUATION_EPSILON,
    "training_summary": full_training_evidence["training"],
}
video_summary = generate_assault_demo_video(
    agent=compact_agent,
    env_factory=lambda: create_assault_env(config, mode="eval", seed=seed + 30_000, render_mode="rgb_array"),
    output_path=VIDEO_PATH,
    metadata=video_metadata,
    seed=seed + 30_000,
    epsilon=EVALUATION_EPSILON,
    max_steps=int(os.environ.get("ASSAULT_VIDEO_MAX_STEPS", "1800")),
    fps=30,
)
print(video_summary.as_dict())
if VIDEO_PATH.exists() and VIDEO_PATH.stat().st_size > 0:
    print("VIDEO_READY=True")
    print("video_path=", VIDEO_PATH)
    print("video_reward=", video_summary.reward)
    print("video_steps=", video_summary.steps)
    print("video_seed=", video_summary.seed)
    print("video_epsilon=", video_summary.epsilon)
    print("video_project_run_id=", video_summary.project_run_id)
    print("video_model_sha256=", video_summary.model_sha256)
    try:
        display(Video(str(VIDEO_PATH), embed=True))
    except Exception as exc:
        print("VIDEO_INLINE_WARNING:", exc)
        print("Video disponible en:", VIDEO_PATH)
else:
    print("VIDEO_READY=False")


## 15. Reporte tecnico academico

### Problema y objetivo
Assault es un entorno Atari visual. El objetivo es maximizar la recompensa raw y evaluar el agente en al menos 10 partidas independientes.

### Seleccion del algoritmo
Se utiliza **DDQN**, método permitido por el enunciado. Es adecuado para un espacio discreto de 7 acciones y observaciones visuales procesadas por CNN. Frente a DQN clásico, separa selección de acción con Online Network y evaluación con Target Network para reducir sobreestimación. No se afirma que sea óptimo frente a todos los métodos permitidos.

### Entorno y preprocessing
`ALE/Assault-v5`, RGB original, grayscale, resize 84x84, stack de 4 frames, frameskip efectivo 4 aplicado una sola vez, `repeat_action_probability=0.25`, `Discrete(7)` y semillas explícitas.

### Arquitectura del agente
CNN Atari-style, Online Network, Target Network, Replay Buffer uniforme, target DDQN y política epsilon-greedy durante entrenamiento; explotación greedy (`epsilon=0.0`) en evaluación.

### Hiperparametros efectivos
Se presentan programáticamente desde el perfil `full`: learning rate, gamma, batch size, Replay Buffer, learning starts, train frequency, Target update, epsilon y 250000 timesteps.

### Librerias, versiones, hardware y tiempo
Las versiones efectivas se obtienen con `get_runtime_info()`. La evidencia versionada de HU009 registra entrenamiento real en Google Colab con NVIDIA A100-SXM4-40GB y tiempo real de entrenamiento.

### Metricas y maximo tres graficas
Las tres figuras de entrenamiento son recompensa + media móvil, loss DDQN y q_mean + epsilon. La figura adicional de explotación corresponde a los episodios finales y no es una cuarta figura de entrenamiento.

### Evaluacion >=10 episodios
El resultado principal se calcula cargando el modelo compacto desde disco, con `epsilon=0.0`, seeds explícitas y al menos 10 episodios independientes.

### Comparacion contra baseline
El baseline aleatorio de HU001 se conserva como JSON versionado derivado de las salidas ejecutadas del experimento 0 y se compara bajo el mismo entorno relevante.

### Comportamiento observado
El video permite observar una política no aleatoria que dispara y se desplaza durante la partida. La interpretación se limita a comportamiento visible y a la mejora cuantitativa frente al baseline; no se atribuyen intenciones internas al agente.

### Limitaciones
Una seed principal de entrenamiento, stochasticity de ALE, evaluación finita, presupuesto de 250000 timesteps y ausencia de HPO exhaustivo. La evaluación histórica del checkpoint se conserva como secundaria; la evaluación principal HU011 corresponde al modelo compacto entregable.

### Conclusion
DDQN se considera efectivo para Assault únicamente si la evaluación final del modelo compacto supera el baseline aleatorio y el video evidencia comportamiento aprendido. El gate HU011 verifica esta condición programáticamente.

### Artefactos de entrega
Notebook, checkpoint fuente, modelo compacto + SHA-256, TensorBoard, evaluación JSON, figura de explotación, baseline versionado, video + metadata y Git SHA.


In [ ]:
hyperparameters = {
    "learning_rate": config["agent"]["learning_rate"],
    "gamma": config["agent"]["gamma"],
    "batch_size": config["replay_buffer"]["batch_size"],
    "replay_buffer_capacity": config["replay_buffer"]["capacity"],
    "learning_starts": config["training"]["learning_starts"],
    "train_frequency": config["training"]["train_frequency"],
    "target_update_frequency": config["training"]["target_update_frequency"],
    "epsilon_start": config["agent"]["epsilon_start"],
    "epsilon_final": config["agent"]["epsilon_final"],
    "epsilon_decay_steps": config["training"]["epsilon_decay_steps"],
    "total_timesteps": config["training"]["total_timesteps"],
}
print("HIPERPARAMETERS=", hyperparameters)
print("RUNTIME_INFO=", runtime_info)
print("TRAINING_EVIDENCE=", full_training_evidence)
print("FINAL_EVALUATION=", compact_evaluation_summary.as_dict())
print("BASELINE=", random_baseline.as_dict())
print("COMPARISON=", baseline_comparison.as_dict())


## 16. Matriz final de cumplimiento del enunciado

El gate siguiente mapea CA01–CA26 de Assault a evidencia ejecutada. CA27 registra el requisito global de usar al menos dos métodos distintos en todo el Reto 1; puede permanecer `PENDING` sin falsear el estado global.


In [ ]:
video_metadata_payload = json.loads(video_summary.metadata_path.read_text(encoding="utf-8"))
TECHNICAL_REPORT_SECTIONS = [
    "Problema y objetivo", "Seleccion del algoritmo", "Entorno y preprocessing",
    "Arquitectura del agente", "Hiperparametros efectivos",
    "Librerias, versiones, hardware y tiempo", "Metricas y maximo tres graficas",
    "Evaluacion >=10 episodios", "Comparacion contra baseline",
    "Comportamiento observado", "Limitaciones", "Conclusion", "Artefactos de entrega",
]
lineage_pass = bool(
    compact_model_info.sha256 == final_evaluation_artifact["model"]["sha256"]
    == video_summary.model_sha256
    and compact_model_info.metadata["project_run_id"] == PROJECT_RUN_ID
    and video_summary.project_run_id == PROJECT_RUN_ID
    and int(compact_model_info.metadata["source_checkpoint_step"]) == CHECKPOINT_STEP
)
statuses = {
    "CA01": True,
    "CA02": _running_in_colab(),
    "CA03": bootstrap.requirements_path.exists(),
    "CA04": COMPACT_MODEL_PATH.exists() and compact_model_info.size_bytes > 0,
    "CA05": lineage_pass,
    "CA06": VIDEO_PATH.exists() and VIDEO_PATH.stat().st_size > 0,
    "CA07": isinstance(video_metadata_payload.get("training_summary"), dict),
    "CA08": video_summary.epsilon == 0.0 and video_summary.steps > 0,
    "CA09": True,
    "CA10": all(value is not None for value in hyperparameters.values()),
    "CA11": all(runtime_info.get(key) for key in ("python_version", "gymnasium_version", "ale_py_version", "torch_version")),
    "CA12": bool(full_training_evidence["hardware"].get("gpu")),
    "CA13": float(full_training_evidence["training"]["duration_seconds"]) > 0,
    "CA14": compact_evaluation_summary.episodes >= 10 and len(set(FINAL_EVAL_SEEDS)) == compact_evaluation_summary.episodes,
    "CA15": all(math.isfinite(value) for value in (compact_evaluation_summary.mean_reward, compact_evaluation_summary.median_reward, compact_evaluation_summary.std_reward, compact_evaluation_summary.min_reward, compact_evaluation_summary.max_reward)),
    "CA16": len(training_figures) == 3 and any(spec.figure_id == "reward" for spec in training_figures),
    "CA17": EXPLOITATION_REWARD_FIGURE_PASS,
    "CA18": baseline_comparison.agent_beats_random,
    "CA19": video_summary.steps > 0 and video_summary.epsilon == 0.0,
    "CA20": baseline_comparison.agent_beats_random,
    "CA21": (ASSAULT_DIR / "assault_ddqn.ipynb").exists() and COMPACT_MODEL_PATH.exists() and VIDEO_PATH.exists(),
    "CA22": all((ASSAULT_DIR / "src" / name).exists() for name in ("agent.py", "environment.py", "evaluator.py", "reporting.py", "video.py")),
    "CA23": DDQNAgent.__name__ == "DDQNAgent" and config["network"]["num_actions"] == 7,
    "CA24": baseline_comparison.agent_beats_random and video_summary.steps > 0,
    "CA25": len(TECHNICAL_REPORT_SECTIONS) == 13,
    "CA26": lineage_pass,
}
evidence = {criterion_id: "evidencia calculada/mostrada en assault_ddqn.ipynb" for criterion_id in MANDATORY_ASSAULT_CRITERIA}
criteria = criteria_from_statuses(statuses, evidence=evidence)
delivery_gate = build_delivery_gate(criteria, global_multi_algorithm=os.environ.get("GLOBAL_RETO_MULTI_ALGORITHM", "PENDING"))
for row in delivery_gate.criteria:
    print(f"{row.criterion_id}={row.status} | {row.evidence}")
print("ASSAULT_METHOD_ALLOWED=", delivery_gate.as_dict()["ASSAULT_METHOD_ALLOWED"])
print("COLAB_NOTEBOOK_EXECUTABLE=", "PASS" if statuses["CA02"] else "FAIL")
print("DEPENDENCIES_PASS=", "PASS" if statuses["CA03"] else "FAIL")
print("MODEL_ARTIFACT_PASS=", "PASS" if statuses["CA04"] else "FAIL")
print("FINAL_EVALUATION_N_GE_10=", "PASS" if statuses["CA14"] else "FAIL")
print("TRAINING_REWARD_FIGURE=", "PASS" if statuses["CA16"] else "FAIL")
print("EXPLOITATION_REWARD_FIGURE=", "PASS" if statuses["CA17"] else "FAIL")
print("RANDOM_BASELINE_COMPARISON=", "PASS" if statuses["CA18"] else "FAIL")
print("ARTIFACT_LINEAGE=", "PASS" if statuses["CA26"] else "FAIL")
print("GLOBAL_RETO_MULTI_ALGORITHM=", delivery_gate.global_multi_algorithm)
print("HU011_FINAL_DELIVERY_GATE=", delivery_gate.as_dict()["HU011_FINAL_DELIVERY_GATE"])
if not delivery_gate.final_delivery_gate:
    print("VALIDACION COLAB PENDIENTE: uno o mas criterios obligatorios de Assault no estan en PASS.")
    raise AssertionError(delivery_gate.as_dict())
print("ENTREGABLE_ASSAULT_LISTO=True")
delivery_gate.as_dict()
